# Week 2 — Frame Your Lane as an ML Task

**Author:** Zain-ul-Abdeen  
**Date:** 2026-08-17  
**Repo:** [Zain-ul-abdeen-773/flyrank-ml-internship](https://github.com/Zain-ul-abdeen-773/flyrank-ml-internship)  
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring  
**Assignment:** ML-03 · Frame Your Lane as an ML Task

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Zain-ul-abdeen-773/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    import subprocess
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir(os.path.join("..", ".."))  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found.")

Working dir: D:\Study\New folder (2)
Starter data found.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns, {df['client_id'].nunique()} clients")

30,000 rows, 44 columns, 32 clients


---
## 1. My Lane as an ML Task (Type)

**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring

**Lane question:** *Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?*

### ML task type: **Scoring → Ranking** (regression-based opportunity scoring)

This is a **scoring and ranking** task. The core operation is:

1. **Regression**: predict an *expected CTR* for each page given its position, content type, intent, volume, and engagement signals.
2. **Gap scoring**: compute the difference `expected_ctr − actual_ctr` to get an *opportunity score* (how many percentage points of CTR the page is leaving on the table).
3. **Ranking**: sort pages by opportunity score (largest gap first, weighted by impression volume) to produce a prioritized review queue.

It is **not classification** (we are not predicting a yes/no label like "will this page decline?"). It is **not clustering** (we are not grouping pages into archetypes). It is a **scoring** task: every page gets a continuous score, and the output is a ranked list ordered by that score.

The closest ML primitive is **regression** (predicting expected CTR as a continuous value), but the *business deliverable* is a **ranked queue** — so the evaluation focuses on ranking quality (are the biggest real opportunities at the top?), not on minimizing squared error.

---
## 2. Target or Proxy

### Target: `ctr_gap` — the gap between expected and actual CTR

There is no single column in the raw data that serves as the target. Instead, the target is **derived**:

```
ctr_gap = expected_ctr(position, content_type, intent, ...) − actual_ctr
```

- A **positive `ctr_gap`** means the page is under-performing — it gets less CTR than comparable pages at the same position. This is the opportunity.
- A **zero or negative `ctr_gap`** means the page is performing at or above expectations.

### Why this is a proxy, not ground truth

The *true* target would be: "if we improve this page's title/meta/content, how much CTR will it gain?" That requires a causal experiment (A/B test). We cannot observe that in this data.

Instead, `ctr_gap` is a **proxy**: it identifies pages that *look like* they should be getting more clicks based on their observable features. A large gap is a *signal worth investigating*, not a guarantee of improvement.

### Simple baseline target (for this notebook)

As a concrete starting point, we define the target using the **position-tier median CTR** as expected CTR. This is the simplest position-adjusted benchmark. A model would later replace this with a learned expected CTR from multiple features.

In [3]:
# ── Filter to visible pages (enough impressions to make CTR meaningful) ──
MIN_IMPRESSIONS = 100
visible = df[(df["impressions_90d"] >= MIN_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
print(f"Visible pages (impressions >= {MIN_IMPRESSIONS}, position > 0): {len(visible):,} of {len(df):,}")

# ── Compute expected CTR per position tier (baseline: tier median) ──
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
print("\nBaseline expected CTR (median per position tier):")
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
print(tier_median_ctr.reindex(tier_order).round(3).to_string())

# ── Build the target: ctr_gap = expected - actual ──
visible["expected_ctr"] = visible["position_tier"].map(tier_median_ctr)
visible["ctr_gap"] = visible["expected_ctr"] - visible["ctr"]

# ── Weight by impressions to prioritize high-volume pages ──
visible["weighted_opportunity"] = visible["ctr_gap"].clip(lower=0) * np.log1p(visible["impressions_90d"])

print(f"\nPages with positive CTR gap (under-performers): {(visible['ctr_gap'] > 0).sum():,}")
print(f"Pages at or above expected CTR: {(visible['ctr_gap'] <= 0).sum():,}")
print(f"\nctr_gap summary statistics:")
print(visible["ctr_gap"].describe().round(3).to_string())

Visible pages (impressions >= 100, position > 0): 22,006 of 30,000

Baseline expected CTR (median per position tier):
position_tier
top_3       0.19
page_1      0.23
striking    0.15
page_3_5    0.06
deep        0.00

Pages with positive CTR gap (under-performers): 10,307
Pages at or above expected CTR: 11,699

ctr_gap summary statistics:
count    22006.000
mean        -0.106
std          0.391
min        -11.530
25%         -0.180
50%          0.000
75%          0.070
max          0.230


---
## 3. Success Metric

### Primary metric: **Precision@K** (K = 50)

The business question is: "if a content reviewer looks at the top 50 pages in our ranked queue, how many of them are *real* opportunities?" Precision@50 measures exactly that.

A "real opportunity" for evaluation purposes is defined as a page that:
- has `ctr_gap > 0` (under-performing its position tier), **and**
- has `impressions_90d >= 500` (enough volume that the gap is meaningful, not noise).

### Secondary metrics

| Metric | What it measures | Why it matters |
|---|---|---|
| **NDCG@50** | Whether higher-opportunity pages are ranked higher than lower-opportunity ones | Captures ranking *quality*, not just hit/miss |
| **Coverage** (recall among top-200) | Of all true high-opportunity pages, how many appear in the top 200? | Ensures we don't miss large opportunities |
| **Weighted opportunity captured** | Total `weighted_opportunity` in the top K vs. total available | Measures business impact: are we surfacing the pages with the most click potential? |

### Why Precision@K, not accuracy or AUC

- **Accuracy** is meaningless here — this is a ranking task, not a balanced classification. Most pages are fine; we only care about the top of the queue.
- **AUC** evaluates the full ranking, but the reviewer only sees the top K. Precision@K matches the actual workflow.
- **RMSE** on CTR prediction matters for the regression step, but the *business* success is whether the ranked queue surfaces real opportunities — that's Precision@K.

### Baseline target for the metric

The naive baseline (sort by raw `ctr` ascending) ignores position entirely. In Week 1 we showed that position-unadjusted CTR comparisons are misleading. If a position-adjusted scoring model achieves Precision@50 ≥ 0.70 (at least 35 of 50 top-ranked pages are genuine opportunities), that would represent a meaningful improvement over the naive sort.

In [4]:
# ── Define what counts as a "true opportunity" for evaluation ──
visible["is_true_opportunity"] = (
    (visible["ctr_gap"] > 0) & (visible["impressions_90d"] >= 500)
).astype(int)

print(f"True opportunities (gap > 0, impressions >= 500): {visible['is_true_opportunity'].sum():,}")
print(f"Share of visible pages: {visible['is_true_opportunity'].mean():.1%}")

# ── Naive baseline: sort by raw CTR ascending (lowest CTR first) ──
naive_sorted = visible.sort_values("ctr", ascending=True)
naive_p50 = naive_sorted.head(50)["is_true_opportunity"].mean()

# ── Position-adjusted baseline: sort by ctr_gap descending (biggest gap first) ──
gap_sorted = visible.sort_values("ctr_gap", ascending=False)
gap_p50 = gap_sorted.head(50)["is_true_opportunity"].mean()

# ── Weighted baseline: sort by weighted_opportunity descending ──
weighted_sorted = visible.sort_values("weighted_opportunity", ascending=False)
weighted_p50 = weighted_sorted.head(50)["is_true_opportunity"].mean()

print(f"\nPrecision@50 comparison:")
print(f"  Naive (sort by raw CTR asc):                {naive_p50:.2f}")
print(f"  Position-adjusted (sort by ctr_gap desc):   {gap_p50:.2f}")
print(f"  Weighted (gap × log impressions):           {weighted_p50:.2f}")
print(f"\nEven a simple position adjustment changes what rises to the top.")

True opportunities (gap > 0, impressions >= 500): 7,159
Share of visible pages: 32.5%



Precision@50 comparison:
  Naive (sort by raw CTR asc):                0.34
  Position-adjusted (sort by ctr_gap desc):   0.36
  Weighted (gap × log impressions):           1.00

Even a simple position adjustment changes what rises to the top.


---
## 4. The Unit of Analysis, as a Real Dataframe

**One row = one content page** (`content_id`), filtered to pages where CTR is actually measurable:
- `impressions_90d >= 100` (enough volume for CTR to be meaningful)
- `avg_position > 0` (has real position data, not the "no data" zero placeholder)

Each row carries:
- **Identifiers**: `content_id`, `client_id`
- **Position context**: `avg_position`, `position_tier`
- **Volume signals**: `impressions_90d`, `clicks_90d`, `sessions_90d`
- **CTR** (the actual observed rate, ×100 scale)
- **Content features**: `content_type`, `main_intent`, `word_count`, `content_age_days`
- **Engagement signals**: `engagement_rate`, `scroll_rate`
- **Derived target**: `expected_ctr`, `ctr_gap`, `weighted_opportunity`

In [5]:
# ── The unit of analysis: one row = one content page ──
display_cols = [
    "content_id", "client_id",
    "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr",
    "content_type", "main_intent",
    "word_count", "content_age_days",
    "engagement_rate", "scroll_rate",
    "expected_ctr", "ctr_gap", "weighted_opportunity"
]

# Show top-10 opportunity pages (biggest CTR gap, weighted by volume)
top_opps = visible.sort_values("weighted_opportunity", ascending=False).head(10)
print("Top 10 CTR opportunity pages (unit of analysis — one row = one page):")
print("="*90)
top_opps[display_cols].to_string(index=False)

Top 10 CTR opportunity pages (unit of analysis — one row = one page):


'          content_id         client_id position_tier  avg_position  impressions_90d  clicks_90d  ctr    content_type   main_intent  word_count  content_age_days  engagement_rate  scroll_rate  expected_ctr  ctr_gap  weighted_opportunity\ncontent_c8e9d6ab9013 client_19581e27de        page_1           9.7           208678           0 0.00 keyword article informational         NaN               362             0.00         0.00          0.23     0.23              2.817167\ncontent_453722754fea client_f369cb89fc        page_1           7.6           140079          16 0.01 keyword article informational      2700.0                97             0.00        11.11          0.23     0.22              2.606993\ncontent_39881853ef0c client_f369cb89fc        page_1           7.2           112434          10 0.01 keyword article informational      2810.0                97             3.45        15.38          0.23     0.22              2.558629\ncontent_c84a0ab98e90 client_f369cb89fc        page_

In [6]:
# Display as a formatted table for readability
top_opps[display_cols]

,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,content_type,main_intent,word_count,content_age_days,engagement_rate,scroll_rate,expected_ctr,ctr_gap,weighted_opportunity
7445,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,208678,0,0.00,keyword article,informational,NaN,362,0.00,0.00,0.23,0.23,2.817167
27178,content_453722754fea,client_f369cb89fc,page_1,7.6,140079,16,0.01,keyword article,informational,2700.0,97,0.00,11.11,0.23,0.22,2.606993
482,content_39881853ef0c,client_f369cb89fc,page_1,7.2,112434,10,0.01,keyword article,informational,2810.0,97,3.45,15.38,0.23,0.22,2.558629
6903,content_c84a0ab98e90,client_f369cb89fc,page_1,7.8,223271,70,0.03,keyword article,informational,2871.0,95,3.45,25.00,0.23,0.20,2.463229
15914,content_0919dd345d80,client_4e07408562,page_1,7.0,119217,26,0.02,keyword article,informational,2841.0,326,9.09,8.70,0.23,0.21,2.454629
13631,content_d274ac4158ef,client_4e07408562,page_1,6.8,65138,6,0.01,keyword article,informational,1428.0,327,4.00,6.25,0.23,0.22,2.438541
24866,content_e5f459e737b7,client_f369cb89fc,page_1,5.9,56363,3,0.01,keyword article,transactional,2916.0,147,0.00,37.50,0.23,0.22,2.406709
4589,content_339b357d04c7,client_bbb965ab0c,page_1,3.7,46879,7,0.01,keyword article,informational,2484.0,133,0.00,14.29,0.23,0.22,2.366176
9193,content_c1fe78bc4e37,client_19581e27de,page_1,7.5,134055,43,0.03,keyword article,commercial,NaN,153,1.18,4.64,0.23,0.20,2.361203
2382,content_65114d89496d,client_19581e27de,page_1,6.5,72631,12,0.02,keyword article,transactional,NaN,482,0.00,0.00,0.23,0.21,2.350564


In [7]:
# ── Sketch what the target column looks like ──
print("Target column sketch: ctr_gap")
print("="*50)
print(f"\nDistribution of ctr_gap across {len(visible):,} visible pages:\n")
print(visible["ctr_gap"].describe().round(3).to_string())

print("\n--- Breakdown by sign ---")
print(f"  Under-performers (gap > 0):   {(visible['ctr_gap'] > 0).sum():>6,}  ({(visible['ctr_gap'] > 0).mean():.1%})")
print(f"  At expected     (gap == 0):    {(visible['ctr_gap'] == 0).sum():>6,}  ({(visible['ctr_gap'] == 0).mean():.1%})")
print(f"  Over-performers (gap < 0):    {(visible['ctr_gap'] < 0).sum():>6,}  ({(visible['ctr_gap'] < 0).mean():.1%})")

print("\n--- Sample rows showing the target ---")
sample = visible.sample(8, random_state=42)[["content_id", "position_tier", "ctr", "expected_ctr", "ctr_gap", "impressions_90d"]]
print(sample.to_string(index=False))

Target column sketch: ctr_gap

Distribution of ctr_gap across 22,006 visible pages:

count    22006.000
mean        -0.106
std          0.391
min        -11.530
25%         -0.180
50%          0.000
75%          0.070
max          0.230

--- Breakdown by sign ---
  Under-performers (gap > 0):   10,307  (46.8%)
  At expected     (gap == 0):     1,110  (5.0%)
  Over-performers (gap < 0):    10,589  (48.1%)

--- Sample rows showing the target ---
          content_id position_tier  ctr  expected_ctr  ctr_gap  impressions_90d
content_69c2c14226b8        page_1 0.30          0.23    -0.07            36901
content_f51e681dcb3e        page_1 0.49          0.23    -0.26              812
content_facb4a134246      page_3_5 0.10          0.06    -0.04             2949
content_62270d7389a7      striking 0.32          0.15    -0.17             4443
content_7e9c6b3fa64c      striking 0.35          0.15    -0.20             5699
content_1be1c6f083ce        page_1 0.89          0.23    -0.66          

---
## 5. Why ML Beats a Fixed Rule Here

### The best fixed rule: "flag all pages with CTR below X%"

A fixed threshold (e.g., `ctr < 0.5`) treats every page the same regardless of position, content type, intent, volume, or engagement context. This fails in three specific, demonstrable ways:

**Problem 1: Position confounds CTR.** A CTR of 0.3% is *terrible* for a top-3 page but *normal* for a page at position 40. A flat rule either floods the queue with deep-position pages that are performing fine, or misses top-position pages that are genuinely under-performing. We showed this in Week 1:

| Position tier | Median CTR |
|---|---|
| top_3 | 0.19 |
| page_1 | 0.23 |
| striking | 0.15 |
| page_3_5 | 0.06 |
| deep | 0.00 |

A fixed CTR < 0.5 rule would flag almost all `page_3_5` and `deep` pages (median CTR already below 0.5) while missing many under-performing `page_1` pages that have CTR of 0.10 (below their tier's 0.23 median, but above 0.5).

**Problem 2: Content type shifts expected CTR *within the same position*.** Comparison articles at page-1 positions have a median CTR of 0.00, while keyword articles at the same position have a median CTR of 0.23. No single threshold handles both.

**Problem 3: Volume matters for prioritization.** A page with 50,000 impressions and a small CTR gap represents more unrealized clicks than a page with 200 impressions and a large gap. A fixed rule cannot rank by expected impact.

### What a model can do that a rule cannot

A model (even a simple one like a decision tree or linear regression) can learn the **multi-dimensional expected-CTR surface**: what CTR *should* look like given position + content type + intent + volume + engagement + freshness, simultaneously. The residual (actual − expected) is a much more honest measure of opportunity than any single-variable threshold.

The model also naturally produces a **continuous score** for ranking, rather than a binary flag. This lets the content team control how deep into the queue they review, based on their available capacity.

### The action the output supports

The ranked queue tells a content/SEO team: *review these pages first — they have the most unrealized click potential given where they already rank.* For each page, the reviewer can:
- **Rewrite the title tag or meta description** to better match searcher intent
- **Improve snippet-eligible structured content** (headings, FAQ blocks, lists)
- **Check intent mismatch** — is the page targeting informational queries but written transactionally?
- **Improve on-page engagement** if engagement/scroll signals also look weak
- **Monitor** if the gap is small or the volume is low

These are cheap, reversible actions. The worst case of a wrong recommendation is a reviewer spends a few minutes checking a page that turns out to be fine.

In [8]:
# ── Demonstrate why a fixed rule fails ──

# Fixed rule: flag everything with CTR < 0.5
rule_flagged = visible[visible["ctr"] < 0.5].copy()

print("Fixed rule: CTR < 0.5")
print(f"  Pages flagged: {len(rule_flagged):,} ({len(rule_flagged)/len(visible):.1%} of visible)")
print(f"  That's too many pages for any team to review.")
print()

# Show what the fixed rule surfaces vs. what the gap-based scoring surfaces
print("Position tier breakdown of top 50 pages:")
print("="*60)

rule_top50 = rule_flagged.sort_values("ctr").head(50)
gap_top50 = visible.sort_values("weighted_opportunity", ascending=False).head(50)

print("\nFixed rule (lowest CTR first):")
print(rule_top50["position_tier"].value_counts().to_string())
print(f"  Median impressions: {rule_top50['impressions_90d'].median():,.0f}")

print("\nGap-based scoring (biggest opportunity first):")
print(gap_top50["position_tier"].value_counts().to_string())
print(f"  Median impressions: {gap_top50['impressions_90d'].median():,.0f}")

print("\n→ The fixed rule surfaces mostly deep/low-volume pages.")
print("→ The gap-based scoring surfaces high-position, high-volume under-performers.")
print("→ That's the difference between noise and actionable opportunity.")

Fixed rule: CTR < 0.5


  Pages flagged: 18,614 (84.6% of visible)
  That's too many pages for any team to review.

Position tier breakdown of top 50 pages:



Fixed rule (lowest CTR first):
position_tier
page_3_5    18
striking    17
page_1       9
deep         5
top_3        1
  Median impressions: 386

Gap-based scoring (biggest opportunity first):
position_tier
page_1    49
top_3      1
  Median impressions: 34,232

→ The fixed rule surfaces mostly deep/low-volume pages.
→ The gap-based scoring surfaces high-position, high-volume under-performers.
→ That's the difference between noise and actionable opportunity.


In [9]:
# ── Content type shifts expected CTR within the same position tier ──
page1 = visible[visible["position_tier"].isin(["page_1", "top_3"])]
ctr_by_type = page1.groupby("content_type")["ctr"].agg(["median", "mean", "count"])

print("CTR by content type (page-1 and top-3 pages only):")
print(ctr_by_type.round(3).to_string())
print()
print("A comparison article with CTR = 0.10 is ABOVE its type's median.")
print("A keyword article with CTR = 0.10 is BELOW its type's median.")
print("No single threshold handles both — the model needs content type as a feature.")

CTR by content type (page-1 and top-3 pages only):
                    median   mean  count
content_type                            
comparison article    0.00  0.141    227
feedly article        0.34  0.949    226
keyword article       0.23  0.344   8713

A comparison article with CTR = 0.10 is ABOVE its type's median.
A keyword article with CTR = 0.10 is BELOW its type's median.
No single threshold handles both — the model needs content type as a feature.


---
## 6. Self-Check

| Check | Answer |
|---|---|
| **Did I name the ML task type?** | ✅ Scoring → Ranking (regression-based opportunity scoring). Predict expected CTR, compute the gap, rank by gap × volume. |
| **Did I name the target or proxy?** | ✅ `ctr_gap` = expected CTR (from position/content/intent features) − actual CTR. This is a proxy for "unrealized click potential." |
| **Did I name the success metric?** | ✅ Precision@50 — of the top 50 pages surfaced, how many are genuine opportunities (gap > 0 with volume ≥ 500 impressions). Secondary: NDCG@50, coverage, weighted opportunity captured. |
| **Did I show the unit of analysis as a real dataframe?** | ✅ One row = one content page (`content_id`), filtered to visible pages. Shown with all key columns including the derived target. |
| **Did I explain why ML beats a fixed rule?** | ✅ Three specific failures of fixed rules (position confounding, content-type variation, volume-based prioritization), with code demonstrating each. |
| **Does the output tie to a real content action?** | ✅ Ranked review queue → rewrite title/meta, improve snippet, check intent match, improve engagement, or monitor. Cheap, reversible actions. |
| **Am I using careful language?** | ✅ All findings described as observed, measured, directional. No causal claims. The gap is a signal worth investigating, not a guarantee of improvement. |
| **Is there leakage?** | ✅ No. `trend_direction` and `trend_pct` are not used as features. The target (`ctr_gap`) is derived from currently-observable metrics (CTR, position), not from future data. |

---
## Summary: The ML Loop for Lane 4

```
┌─────────────────────────────────────────────────────────────┐
│  TASK TYPE:   Scoring → Ranking (regression-based)         │
│  TARGET:      ctr_gap = expected_ctr − actual_ctr          │
│  METRIC:      Precision@50 (is the top of the queue good?) │
│  UNIT:        One content page (content_id)                │
│  ACTION:      Ranked review queue → rewrite / improve /    │
│               check intent / monitor                       │
│  WHY ML:      Position, content type, intent, volume all   │
│               interact — no single rule captures it        │
└─────────────────────────────────────────────────────────────┘
```